<a href="https://colab.research.google.com/github/imostafizur/tensorflow_board/blob/master/MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Dataset Download

In [1]:
import os

os.environ['KAGGLE_USERNAME'] = "imostafizur" # username from the json file
os.environ['KAGGLE_KEY'] = "209f4b70995998e465b2702534f7a5ff" # key from the json file

In [2]:
!kaggle datasets download ruchi798/movies-on-netflix-prime-video-hulu-and-disney

Dataset URL: https://www.kaggle.com/datasets/ruchi798/movies-on-netflix-prime-video-hulu-and-disney
License(s): CC0-1.0
  0% 0.00/166k [00:00<?, ?B/s]
100% 166k/166k [00:00<00:00, 430MB/s]


In [3]:
!unzip movies-on-netflix-prime-video-hulu-and-disney.zip

Archive:  movies-on-netflix-prime-video-hulu-and-disney.zip
  inflating: MoviesOnStreamingPlatforms.csv  


## Dataset view

### Python Standard Library

In [4]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import re, os

### Dataset loading

In [5]:
file_path = "/content/MoviesOnStreamingPlatforms.csv"
assert os.path.exists(file_path), f"File not found: {file_path}"
df = pd.read_csv(file_path)
print("Initial shape:", df.shape)

Initial shape: (9515, 11)


In [6]:
# Show the first few rows
df.head()

,Unnamed: 0,ID,Title,Year,Age,Rotten Tomatoes,Netflix,Hulu,Prime Video,Disney+,Type
0,0,1,The Irishman,2019,18+,98/100,1,0,0,0,0
1,1,2,Dangal,2016,7+,97/100,1,0,0,0,0
2,2,3,David Attenborough: A Life on Our Planet,2020,7+,95/100,1,0,0,0,0
3,3,4,Lagaan: Once Upon a Time in India,2001,7+,94/100,1,0,0,0,0
4,4,5,Roma,2018,18+,94/100,1,0,0,0,0


In [7]:
df.columns

Index(['Unnamed: 0', 'ID', 'Title', 'Year', 'Age', 'Rotten Tomatoes',
       'Netflix', 'Hulu', 'Prime Video', 'Disney+', 'Type'],
      dtype='object')

### Drop useless columns

In [8]:
for c in ['Unnamed: 0', 'ID']:
    if c in df.columns:
        df = df.drop(columns=[c])

print("After dropping index/ID columns:", df.shape)

After dropping index/ID columns: (9515, 9)


### HANDLE TYPE COLUMN SAFELY

In [9]:
# Filter only movies (Type == 0) — dataset uses 0 = movie, 1 = show
if "Type" in df.columns:
    print("Type column values:", df["Type"].value_counts(dropna=False))

    # Convert to numeric safely
    type_num = pd.to_numeric(df["Type"], errors="coerce")

    if set(type_num.dropna().unique()).issubset({0.0, 1.0}):
        df = df[type_num == 0].copy()
        print("Filtered movies only (Type == 0).")
    else:
        print("Unknown Type format — no filtering applied.")

print("After Type filtering:", df.shape)

Type column values: Type
0    9515
Name: count, dtype: int64
Filtered movies only (Type == 0).
After Type filtering: (9515, 9)


### Parse Rotten Tomatoes -> numeric percent


In [10]:
def parse_rt_robust(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "" or s.lower() in ("nan", "none", "n/a"):
        return np.nan

    # Formats like "98/100"
    m = re.search(r"^\s*(\d{1,3})\s*/\s*100\s*$", s)
    if m:
        return float(m.group(1))

    # Formats like "94%"
    m = re.search(r"^(\d{1,3})\s*%$", s)
    if m:
        return float(m.group(1))

    # Plain numbers
    m = re.search(r"(\d{1,3})", s)
    if m:
        return float(m.group(1))

    return np.nan

df["rt_score"] = df["Rotten Tomatoes"].apply(parse_rt_robust)
print("\nRT parsing complete. Example:")
print(df[["Rotten Tomatoes", "rt_score"]].head(8))


RT parsing complete. Example:
  Rotten Tomatoes  rt_score
0          98/100      98.0
1          97/100      97.0
2          95/100      95.0
3          94/100      94.0
4          94/100      94.0
5          94/100      94.0
6          93/100      93.0
7          92/100      92.0


### Convert Age -> ordinal code

In [11]:
# Cell 5 - Convert Age to ordinal code (0..4) for "lower age restriction" comparisons
def age_to_code(age):
    if pd.isna(age): return np.nan
    s = str(age).strip().lower()
    if s in ("all","all ages"): return 0
    m = re.search(r"(\d+)", s)
    if not m: return np.nan
    n = int(m.group(1))
    if n <= 7: return 1
    if n <= 13: return 2
    if n <= 16: return 3
    return 4

df['age_code'] = df['Age'].apply(age_to_code)
print("age_code distribution:")
print(df['age_code'].value_counts(dropna=False).sort_index())

age_code distribution:
age_code
0.0     698
1.0    1090
2.0     998
3.0     276
4.0    2276
NaN    4177
Name: count, dtype: int64


### Clean Platfrom Flags

In [12]:
# Cell 6 - Platform flags (numeric -> int). Works in Colab & local.
for col in ["Netflix", "Hulu", "Prime Video", "Disney+"]:
    if col in df.columns:
        df[col + "_flag"] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
    else:
        df[col + "_flag"] = 0
display(df[['Netflix_flag','Disney+_flag','Hulu_flag','Prime Video_flag']].head(4))

,Netflix_flag,Disney+_flag,Hulu_flag,Prime Video_flag
0,1,0,0,0
1,1,0,0,0
2,1,0,0,0
3,1,0,0,0


### Year numeric & drop duplicates

In [13]:
# Cell 7 - Year numeric and drop duplicates
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
removed = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Removed {removed} duplicate rows. New shape: {df.shape}")

Removed 0 duplicate rows. New shape: (9515, 15)


### Data Quality Summary

In [14]:
# Final quick checks
print("Final dataset shape:", df.shape)
print("Missing rt_score:", df['rt_score'].isna().sum())
print("Missing age_code:", df['age_code'].isna().sum())
print("Missing Year:", df['Year'].isna().sum())
print("\nPlatform counts (0/1):")
for col in ['Netflix_flag','Disney+_flag','Hulu_flag','Prime Video_flag']:
    print(f"  {col}: {df[col].value_counts(dropna=False).to_dict()}")

Final dataset shape: (9515, 15)
Missing rt_score: 7
Missing age_code: 4177
Missing Year: 0

Platform counts (0/1):
  Netflix_flag: {0: 5820, 1: 3695}
  Disney+_flag: {0: 8593, 1: 922}
  Hulu_flag: {0: 8468, 1: 1047}
  Prime Video_flag: {0: 5402, 1: 4113}


### Data Quality Summary

In [15]:
print("\n=== Final Data Quality Summary ===")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\nMissing Values:")
for col in ["rt_score", "age_code", "Year"]:
    print(f"  {col}: {df[col].isna().sum()}")

print("\nPlatform Availability (counts):")
for col in [c for c in df.columns if c.endswith("_flag")]:
    counts = df[col].value_counts(dropna=False).to_dict()
    print(f"  {col}: {counts}")



=== Final Data Quality Summary ===
Rows: 9515, Columns: 15

Missing Values:
  rt_score: 7
  age_code: 4177
  Year: 0

Platform Availability (counts):
  Netflix_flag: {0: 5820, 1: 3695}
  Hulu_flag: {0: 8468, 1: 1047}
  Prime Video_flag: {0: 5402, 1: 4113}
  Disney+_flag: {0: 8593, 1: 922}


### Save clean dataset

In [16]:
out_path = "/content/movies_cleaned.csv"
df.to_csv(out_path, index=False)
print("\nCleaned dataset saved to:", out_path)


Cleaned dataset saved to: /content/movies_cleaned.csv


In [17]:
# Cell 4 - Parse Rotten Tomatoes (robust but simple)
import re

def parse_rt_robust(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    m = re.search(r"(\d{1,3})", s)
    return float(m.group(1)) if m else np.nan

df['rt_score'] = df['Rotten Tomatoes'].apply(parse_rt_robust)
print("rt_score missing:", df['rt_score'].isna().sum())
display(df[['Rotten Tomatoes','rt_score']].head(6))

rt_score missing: 7


,Rotten Tomatoes,rt_score
0,98/100,98.0
1,97/100,97.0
2,95/100,95.0
3,94/100,94.0
4,94/100,94.0
5,94/100,94.0


## Two-sample t-test (pooled variance)

**Pooled Variance:**
$$s_p^2 = \frac{(n_x-1) s_x^2 + (n_y-1) s_y^2}{n_x + n_y - 2}$$

**t-statistic:**
$$t = \frac{\bar{X} - \bar{Y}}{s_p \sqrt{\frac{1}{n_x} + \frac{1}{n_y}}}$$

**Degrees of Freedom:**
$$df = n_x + n_y - 2$$

---

### Key to Variables:
* $\bar{X}, \bar{Y}$: Sample means
* $s_x^2, s_y^2$: Sample variances
* $n_x, n_y$: Sample sizes
* $s_p$: Pooled standard deviation


In [18]:
# Cell 10 - Manual pooled t-test implementation (educational)
def pooled_t_test(x, y):
    x = np.asarray(x[~np.isnan(x)])
    y = np.asarray(y[~np.isnan(y)])
    n_x, n_y = len(x), len(y)
    mean_x, mean_y = x.mean(), y.mean()
    var_x, var_y = x.var(ddof=1), y.var(ddof=1)
    sp2 = ((n_x - 1) * var_x + (n_y - 1) * var_y) / (n_x + n_y - 2)
    se = np.sqrt(sp2 * (1.0/n_x + 1.0/n_y))
    t_stat = (mean_x - mean_y) / se
    dfree = n_x + n_y - 2
    p_two = 2 * stats.t.sf(np.abs(t_stat), dfree)
    return {"t": t_stat, "df": dfree, "p_two": p_two, "mean_x": mean_x, "mean_y": mean_y}

## Mann–Whitney U Test — Mathematical Formulation

Given two independent samples:
* $X = \{x_1, x_2, \dots, x_{n_x}\}$
* $Y = \{y_1, y_2, \dots, y_{n_y}\}$

Rank all values from both samples together in ascending order.

### U statistic for group X
$$U_X = \sum_{i=1}^{n_x} R(x_i) - \frac{n_x(n_x+1)}{2}$$

### U statistic for group Y
$$U_Y = \sum_{j=1}^{n_y} R(y_j) - \frac{n_y(n_y+1)}{2}$$

Since $U_X + U_Y = n_x n_y$, the test statistic is usually defined as:
$$U = \min(U_X, U_Y)$$

### Mean and standard deviation under the null hypothesis
$$\mu_U = \frac{n_x n_y}{2}$$

$$\sigma_U = \sqrt{\frac{n_x n_y (n_x + n_y + 1)}{12}}$$

### z-score approximation (for large samples)
$$z = \frac{U - \mu_U}{\sigma_U}$$

### Two-sided p-value
$$p = 2 \Phi(-|z|)$$

---
**Where:**
* $R(x_i), R(y_j)$: The rank of the observation in the combined sample.
* $\Phi$: The standard normal Cumulative Distribution Function (CDF).

In [19]:
# Cell 11 - Mann-Whitney (scipy) wrapper
def mann_whitney(x, y):
    x = np.asarray(x[~np.isnan(x)])
    y = np.asarray(y[~np.isnan(y)])
    u_stat, p_two = stats.mannwhitneyu(x, y, alternative='two-sided')
    return {"U": u_stat, "p_two": p_two, "n_x": len(x), "n_y": len(y)}

## Group Definition for Hypothesis Testing

To compare Netflix and Disney+, we define two samples:

* $X$: Rotten Tomatoes scores of movies available on Netflix
* $Y$: Rotten Tomatoes scores of movies available on Disney+

Optionally, we can define **exclusive** groups:

* $X_{\text{excl}}$: Movies available on Netflix **but not** on Disney+
* $Y_{\text{excl}}$: Movies available on Disney+ **but not** on Netflix

These groups form the input for the statistical tests (e.g., Welch's t-test or Mann–Whitney U).

In [20]:
# Cell 12 - Prepare comparison groups (example)
# Option A: all titles available on the platform
group_netflix_rt = df.loc[df['Netflix_flag']==1, 'rt_score']
group_disney_rt  = df.loc[df['Disney+_flag']==1, 'rt_score']

# Option B: exclusive titles (on A but not B)
group_netflix_excl = df.loc[(df['Netflix_flag']==1) & (df['Disney+_flag']==0), 'rt_score']
group_disney_excl  = df.loc[(df['Disney+_flag']==1) & (df['Netflix_flag']==0), 'rt_score']

print("Counts (all):", len(group_netflix_rt), len(group_disney_rt))
print("Counts (exclusive):", len(group_netflix_excl), len(group_disney_excl))

Counts (all): 3695 922
Counts (exclusive): 3689 916


### Run tests and print results

In [21]:
# Cell 13 - Run statistical tests (choose exclusive or all)
x = group_netflix_excl.dropna()
y = group_disney_excl.dropna()

print("Manual pooled t-test result:")
print(pooled_t_test(x, y))

print("\nscipy ttest_ind (equal_var=True):")
print(stats.ttest_ind(x, y, equal_var=True, nan_policy='omit'))

print("\nMann-Whitney result:")
print(mann_whitney(x, y))

Manual pooled t-test result:
{'t': np.float64(-7.542774326625193), 'df': 4596, 'p_two': np.float64(5.512164038555537e-14), 'mean_x': np.float64(54.43970668115155), 'mean_y': np.float64(58.30458515283843)}

scipy ttest_ind (equal_var=True):
TtestResult(statistic=np.float64(-7.542774326625193), pvalue=np.float64(5.512164038555537e-14), df=np.float64(4596.0))

Mann-Whitney result:
{'U': np.float64(1404256.5), 'p_two': np.float64(4.212078242214384e-15), 'n_x': 3682, 'n_y': 916}


In [23]:
def cohen_d(x, y):
    """Compute Cohen's d effect size."""
    x = np.asarray(x.dropna())
    y = np.asarray(y.dropna())
    nx, ny = len(x), len(y)
    pooled_var = (((nx-1)*x.var(ddof=1)) + ((ny-1)*y.var(ddof=1))) / (nx+ny-2)
    return (x.mean() - y.mean()) / np.sqrt(pooled_var)

# Choose your groups: (exclusive example)
X = group_netflix_excl.dropna()
Y = group_disney_excl.dropna()

print("==================================================")
print("           COMPARISON OF NETFLIX VS DISNEY+       ")
print("==================================================")

print(f"Number of movies on Netflix (sample X): {len(X)}")
print(f"Number of movies on Disney+ (sample Y): {len(Y)}\n")

print("Descriptive Statistics:")
print(f"  Netflix  - mean: {X.mean():.3f}, median: {X.median():.3f}, std: {X.std():.3f}")
print(f"  Disney+  - mean: {Y.mean():.3f}, median: {Y.median():.3f}, std: {Y.std():.3f}\n")

# --- t-test ---
t_res = stats.ttest_ind(X, Y, equal_var=True, nan_policy='omit')

print("Two-sample t-test (equal variance):")
print(f"  t-statistic: {t_res.statistic:.4f}")
print(f"  p-value:     {t_res.pvalue:.4f}")

# --- Mann-Whitney ---
mw_res = mann_whitney(X, Y)
print("\nMann–Whitney U Test:")
print(f"  U-statistic: {mw_res['U']:.4f}")
print(f"  p-value:     {mw_res['p_two']:.4f}")

# --- Effect size ---
d = cohen_d(X, Y)
print(f"\nEffect Size (Cohen’s d): {d:.4f}")

# --- Interpretation ---
alpha = 0.05
print("\nInterpretation (α = 0.05):")
if t_res.pvalue < alpha:
    print("  ✔ There is statistically significant evidence that the Rotten Tomatoes")
    print("    scores differ between Netflix and Disney+.")
else:
    print("  ✘ There is no statistically significant evidence of a difference in")
    print("    Rotten Tomatoes scores between Netflix and Disney+.")

print("\nNote: Cohen's d describes practical difference:")
print("  ~0.2 small, ~0.5 medium, ~0.8 large effect.")
print("==================================================\n")


           COMPARISON OF NETFLIX VS DISNEY+       
Number of movies on Netflix (sample X): 3682
Number of movies on Disney+ (sample Y): 916

Descriptive Statistics:
  Netflix  - mean: 54.440, median: 53.000, std: 13.852
  Disney+  - mean: 58.305, median: 57.500, std: 13.978

Two-sample t-test (equal variance):
  t-statistic: -7.5428
  p-value:     0.0000

Mann–Whitney U Test:
  U-statistic: 1404256.5000
  p-value:     0.0000

Effect Size (Cohen’s d): -0.2785

Interpretation (α = 0.05):
  ✔ There is statistically significant evidence that the Rotten Tomatoes
    scores differ between Netflix and Disney+.

Note: Cohen's d describes practical difference:
  ~0.2 small, ~0.5 medium, ~0.8 large effect.

